<a href="https://colab.research.google.com/github/jazoza/cultural-data-analysis/blob/main/03_CDA_Europeana_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cultural Data Analysis

Importing and Analysing Europeana data

https://www.europeana.eu/en

In [ ]:
import os
os.environ['EUROPEANA_API_KEY'] = '' # add your personal/project key

In [ ]:
!pip install pyeuropeana --no-deps
!pip install fire

In [ ]:
import pyeuropeana.apis as apis
import pyeuropeana.utils as utils

### A. Get a section/selection of data

In [ ]:
result = apis.search(
    query = '*',
    #qf = '(skos_concept:"http://data.europeana.eu/concept/base/48" AND TYPE:IMAGE)',
    #qf = '(COUNTRY:"United Kingdom" OR COUNTRY:"Ireland")',
    #reusability = 'open AND permission',
    #landingpage = True,
    #media = True,
    #thumbnail = True,
    #colourpalette = '#0000FF',
    #theme = 'photography',
    theme = 'industrial',
    sort = 'europeana_id',
    profile = 'rich',
    rows = 200,
    )

### B. Identify the object you looked at during the class

In [ ]:
result = apis.search(
    query = '*', # YOUR SEARCH TERM(S)
    theme = 'industrial', # LEAVE BLANK '' IF YOU ARE NOT SURE
    sort = 'europeana_id',
    profile = 'rich',
    rows = 20)
df = utils.search2df(result)
df.head()

In [ ]:
#df.loc[0]['uri'] # this will give you the direct link to the 1st ([0]) item
df.columns # this will list all columns in the dataset

In [ ]:
import pandas as pd

def flat(d):
    return list(dict.fromkeys(v for vals in d.values() for v in vals))

def get_subject(item):
    subj = flat(item.get('dcSubjectLangAware', {}))
    if subj:
        return subj, 'provider'
    c = item.get('edmConceptPrefLabelLangAware', {})
    return (c.get('en') or flat(c)), 'europeana concept'

rows = []
for it in result['items']:
    subj, src = get_subject(it)
    rows.append({'europeana_id': it['id'], 'subject': subj, 'subject_source': src})

df = df.merge(pd.DataFrame(rows), on='europeana_id', how='left')

In [ ]:
df.head()

### Data structure

#### Temporal structure

In [ ]:
# time extraction function
def extract_year(x):
    # 1. handle missing values safely (scalar-only check)
    if x is None:
        return "NaN"
    # 2. handle lists explicitly
    if isinstance(x, list):
        if len(x) == 0:
            return "NaN"
        x = x[0]  # take first element
    # 3. convert to string safely
    x = str(x)
    # 4. extract year (first 4 digits if possible)
    digits = "".join(ch for ch in x[:10] if ch.isdigit())
    if len(digits) >= 4:
        return int(digits[:4])
    return "NaN"

In [ ]:
df["year"] = pd.to_numeric(df["edmTimespan"].apply(extract_year), errors="coerce")

In [ ]:
from collections import Counter
import pandas as pd

# timespan analysis: relevant column is 'edmTimespan'
columns = 'edmTimespan'

# global counter
year_counts = Counter()

# Apply the extraction directly to the in-memory dataframe `df`
if columns in df.columns:
    years = df[columns].apply(extract_year)
    year_counts.update(years)
else:
    print(f"Column '{columns}' not found in the DataFrame.")

# Convert all keys to strings to prevent mixed-type comparison during sorting
year_counts = {str(k): v for k, v in year_counts.items()}
year_counts = dict(sorted(year_counts.items()))
dict(list(year_counts.items())[:5])

In [ ]:
import matplotlib.pyplot as plt

# Counter -> Series, drop non-numeric keys, sort
year_series = (
    pd.Series(year_counts)
    .pipe(lambda s: s[pd.to_numeric(s.index, errors="coerce").notna()])
)
year_series.index = year_series.index.astype(int)
year_series = year_series.sort_index()

plt.figure(figsize=(36, 6))

plt.bar(year_series.index.astype(str), year_series.values, color="grey")
plt.xlabel("Year")
plt.ylabel("Count")
plt.title("Distribution of edmTimespan by Year")
plt.xticks(ticks=range(0, len(year_series), 5), labels=year_series.index.astype(str)[::5], rotation=90)
plt.tight_layout()
plt.margins(x=0.01)
plt.savefig("edmTimespan_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# define time periods
bins = [0, 1491, 1789, 1903, 1929, 1955, float("inf")]

labels = [
    "0–1491",
    "1492–1789",
    "1790–1903",
    "1904–1929",
    "1930–1955",
    "1956–today"
]

df["period"] = pd.cut(
    df["year"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
df.head()

In [ ]:
palette = [
    "#DBDAE8",
    "#C1BEDB",
    "#A7A2CE",
    "#8E86C1",
    "#746AB4",
    "#5A4EA7",
    "#463D86"
]

#### Topic

#### 1.3.1 TF-IDF Descriptions analysis

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

vectorizer = TfidfVectorizer(
    token_pattern=r'(?u)\b\w[\w\'\-]*\w\b|\b\w\b',  # Unicode word pattern
    lowercase=True,
    max_df=0.85,          # Ignore terms in >85% of documents
    min_df=3,             # Ignore terms in <3 documents
    max_features=10000,   # Limit vocabulary for memory/speed
    ngram_range=(1, 2),   # Unigrams + bigrams (captures "physical tool")
    sublinear_tf=True,    # Apply log scaling to term frequency
    dtype=np.float32,     # Memory optimization
    norm='l2'             # Standard L2 normalization
)

print("Building TF-IDF matrix (this may take a few minutes)...")
tfidf_matrix = vectorizer.fit_transform(df_clean['description_clean'])

feature_names = vectorizer.get_feature_names_out()
print(f"TF-IDF matrix: {tfidf_matrix.shape[0]:,} docs × {tfidf_matrix.shape[1]} features")
print(f"Memory usage: ~{tfidf_matrix.data.nbytes / 1e6:.1f} MB (sparse format)")

In [ ]:
global_tfidf_scores = np.asarray(tfidf_matrix.sum(axis=0)).flatten()
# Create a DataFrame for clean sorting and viewing
tfidf_df = pd.DataFrame({
    'term': feature_names,
    'global_tfidf_score': global_tfidf_scores
})
# Sort descending and grab the top 100
top_100_terms = (tfidf_df
                 .sort_values(by='global_tfidf_score', ascending=False)
                 .head(100)
                 .reset_index(drop=True))

top_100_terms.index = top_100_terms.index + 1
top_100_terms.index.name = 'Rank'

In [ ]:
with pd.option_context('display.max_rows', None,       # Show all 100 rows
                       'display.max_colwidth', None,   # Do not truncate text strings
                       'display.width', None):         # Prevent line wrapping
    display(top_100_terms)

#### 1.3.2 COLLOCATIONS - Descriptions analysis

In [ ]:
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
import re

def tokenize_simple(text):
    # Find all word-like chunks
    tokens = re.findall(r'\b[\w\'\-]+\b', text.lower())
    # Keep tokens which Do NOT contain any digits AND Contain at least one alphabetic character
    return [t for t in tokens if not any(c.isdigit() for c in t) and any(c.isalpha() for c in t)]
print("Tokenizing descriptions")
corpus_tokens = [tokenize_simple(text) for text in df_clean['description_clean'].tolist()]
print("Extracting significant bigrams")
finder = BigramCollocationFinder.from_documents(corpus_tokens)
# Filter: ignore very rare pairs and enforce text-only at the finder level too
finder.apply_freq_filter(5)  # Increased min frequency to 5 to eliminate random OCR/typo noise
finder.apply_word_filter(lambda w: len(w) < 2 or any(c.isdigit() for c in w))
# Use Likelihood Ratio instead of PMI.
top_bigrams = finder.nbest(BigramAssocMeasures.likelihood_ratio, 40)
print("\n Top multilingual text bigrams:")
for i, (w1, w2) in enumerate(top_bigrams, 1):
    print(f"{i:2d}. {w1} {w2}")

#### 1.3.3. TOPIC MODELING - NMF - Descriptions analysis

In [ ]:
from sklearn.decomposition import NMF

def extract_topics_nmf(tfidf_matrix, feature_names, n_topics=20, top_words=15):
    print(f" Training NMF model with {n_topics} topics...")

    model = NMF(
        n_components=n_topics,
        random_state=42,
        max_iter=400,
        init='nndsvda'
)
    doc_topic_dist = model.fit_transform(tfidf_matrix)

    # Extract top terms per topic
    topics = []
    print(f"\n Top {top_words} terms per topic:")
    print("=" * 80)

    for topic_idx in range(n_topics):
        top_terms_idx = model.components_[topic_idx].argsort()[:-top_words-1:-1]
        top_terms = [feature_names[i] for i in top_terms_idx]
        topics.append(top_terms)

        # Display with simple scoring
        scores = model.components_[topic_idx][top_terms_idx]
        terms_scored = [f"{t}({s:.2f})" for t, s in zip(top_terms, scores)]

        print(f"\nTopic {topic_idx+1:2d}:")
        print("  " + " | ".join(terms_scored))

    return model, doc_topic_dist, topics

# Run topic modeling
topic_model, doc_topics, topic_terms = extract_topics_nmf(
    tfidf_matrix, feature_names, n_topics=25, top_words=12
)

# Assign dominant topic to each document
df_clean['dominant_topic'] = doc_topics.argmax(axis=1) + 1
df_clean['topic_confidence'] = doc_topics.max(axis=1)  # How strongly assigned?

# Topic distribution
topic_counts = df_clean['dominant_topic'].value_counts().sort_index()
print(f"\n Documents per topic:\n{topic_counts.to_string()}")

#### 1.3.4 Entries/Text per country

In [ ]:
def extract_country(val):
    """Safely extract the country string from a list, string, or NaN."""
    if isinstance(val, list) and len(val) > 0:
        return str(val[0])
    elif isinstance(val, str):
        return val
    elif pd.isna(val):
        return 'Unknown'
    else:
        return str(val)

df['country_name'] = df['country'].apply(extract_country)
#entries_per_country = df['country_name'].value_counts().sort_values(ascending=False)
entries_per_country = df['country_name'].value_counts().sort_index()

print(f" Found {len(entries_per_country)} unique countries.\n")
print("Top 10 Countries by Entry Count:")
display(entries_per_country.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

custom_palette = sns.color_palette(["#463E86", "#928DB6"], n_colors=len(entries_per_country))

plt.figure(figsize=(14, 6))
sns.barplot(
    x=entries_per_country.index,
    y=entries_per_country.values,
    palette=custom_palette,
    hue=entries_per_country.index, # Avoids future warnings in seaborn
    legend=False
)

plt.title('Number of Entries per Country', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Number of Entries', fontsize=12)
plt.xticks(rotation=90, ha='right')
plt.tight_layout()
plt.savefig("TOTAL_entriespercountry.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# COUNT NUMBER OF WORDS PER COUNTRY
def count_actual_words(raw_desc):
    # 1. Handle None
    if raw_desc is None:
        return 0
    # 2. Safely handle scalar NaNs (which are stored as floats in Pandas)
    if isinstance(raw_desc, float) and np.isnan(raw_desc):
        return 0
    # 3. Convert lists, numpy arrays, and pandas Series to strings
    if isinstance(raw_desc, (list, np.ndarray, pd.Series)):
        text = " ".join(str(s) for s in raw_desc)
    else:
        text = str(raw_desc)
    # 4. Strip out junk (URLs, brackets, quotes)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
    text = re.sub(r'[\[\]\{\}\'\"]', '', text)          # Remove brackets and quotes
    # 5. Find all actual words (alphanumeric sequences)
    words = re.findall(r'\b\w+\b', text.lower())
    # 6. Filter out the literal word 'nan' just in case a float NaN was hiding inside a list
    words = [w for w in words if w != 'nan']
    return len(words)

df['word_count'] = df['dcDescription'].apply(count_actual_words)
total_words_per_country = df.groupby('country_name')['word_count'].sum()
total_words_per_country = total_words_per_country.sort_index()

In [ ]:
# --- PLOT: Total Words per Country (Alphabetical) ---
custom_palette = sns.color_palette(["#463E86", "#928DB6"], n_colors=len(total_words_per_country))
plt.figure(figsize=(16, 6))
sns.barplot(
    x=total_words_per_country.index,
    y=total_words_per_country.values,
    palette=custom_palette,
    hue=total_words_per_country.index,
    legend=False
)

plt.title('Total Word Count in Descriptions per Country (Alphabetical Order)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Total Words', fontsize=12)
plt.xticks(rotation=90, ha='right')
plt.savefig("TOTAL_wordspercountry.png", dpi=300, bbox_inches="tight")
plt.tight_layout()
plt.show()

#### Countries, institutions

In [ ]:
institution_counts = df['dataProvider'].value_counts().reset_index()
institution_counts.columns = ['dataProvider','count']
print(institution_counts)

In [ ]:
institution_counts = (

    df.assign(
        country=df['country'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x),
        dataProvider=df['dataProvider'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)
    )
    .groupby(['dataProvider', 'country'])
    .size()
    .reset_index(name='count')
    .sort_values('country', ascending=False)
)

print(institution_counts[['dataProvider', 'country', 'count']])

In [ ]:
print(df[df['dataProvider'].apply(lambda x: isinstance(x, list))])

In [ ]:
country_counts = df['country'].value_counts().reset_index()
country_counts.columns = ['country', 'count']
print(country_counts)